[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module0_TimeSeries/12_ExogenousLagMatrices.ipynb#copy=true)

# Exogenous-Aware Lag Matrices

**Module 0 · Lesson 12 of 13 · Student edition**  
**Estimated class time:** 80–95 minutes  
**Source sequence:** Original Day 4  

**Prerequisite:** Lesson 11  

## Learning objectives

By the end of this lesson, you should be able to:

- Build a univariate forecasting baseline on daily data.
- Align same-time exogenous information with lagged target features.
- Compare univariate and exogenous-aware linear models.

## Setup for this lesson

This cell recreates the data and completed prerequisites from earlier lessons, so this notebook can be run in a fresh kernel.

In [ ]:
# Shared setup from the preceding lesson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

url = 'https://raw.githubusercontent.com/christophM/interpretable-ml-book/master/data/bike-sharing-daily.csv'
df = pd.read_csv(url, parse_dates=['dteday'])
df = df[['dteday', 'cnt', 'temp', 'hum', 'windspeed', 'workingday', 'holiday']].copy()
df.columns = ['Date', 'Rentals', 'Temp', 'Humidity', 'Windspeed', 'WorkingDay', 'Holiday']
df['Rentals_Diff'] = df['Rentals'].diff()


***
## Part 3 — Baseline: Univariate Lag Matrix (No Exogenous Variables)

Before adding external variables, build the same kind of univariate lag matrix from
Day 2 — this is today's baseline for comparison.

**Your turn!** Complete `make_lag_matrix` (you've written this before) and build
$(X, y)$ using only `Rentals_Diff`'s own lags.


In [11]:
def make_lag_matrix(series, n_lags):
    series = np.array(series)
    T = len(series)
    X, y = [], []
    for t in range(n_lags, T):
        X.append(series[t - n_lags : t])
        y.append(series[t])
    return np.array(X), np.array(y)

N_LAGS = 7   # one week of history
target_series = df['Rentals_Diff'].dropna().values

X_uni, y_uni = make_lag_matrix(target_series, N_LAGS)
print('Univariate X shape:', X_uni.shape)


Univariate X shape: (723, 7)


> 💡 **Why `N_LAGS=7`?** Bike rentals are daily data, and weekly patterns (weekday
> vs.\ weekend commuting) are a likely source of short-term dependence. This is a
> different — and equally defensible — choice than the `N_LAGS=12` used for Air
> Passengers' monthly seasonality.


### 3.1 Time-aware split (same rule as Day 3)

**Your turn!** Split chronologically — no shuffling — using the last 20% as test data.


In [12]:
TRAIN_FRAC = 0.80

# FILL IN: compute the split index
split = int(len(X_uni) * TRAIN_FRAC)

X_uni_train, X_uni_test = X_uni[:split], X_uni[split:]
y_uni_train, y_uni_test = y_uni[:split], y_uni[split:]

print(f'Training samples: {len(X_uni_train)}')
print(f'Test samples:     {len(X_uni_test)}')


Training samples: 578
Test samples:     145


### 3.2 Fit a baseline model and evaluate

Use `LinearRegression` (the same AR-equivalent model from Day 2) as today's
univariate baseline.


In [13]:
def compute_metrics(y_true, y_pred, label='Model'):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    print(f'{label:<30}  RMSE={rmse:.4f}   MAE={mae:.4f}')
    return {'RMSE': rmse, 'MAE': mae}

results = {}

# FILL IN: fit a LinearRegression on the univariate training data
lr_uni = LinearRegression()
lr_uni.fit(X_uni_train, y_uni_train)
lr_uni_preds = lr_uni.predict(X_uni_test)

results['Univariate (lags only)'] = compute_metrics(y_uni_test, lr_uni_preds, 'Univariate Linear')


Univariate Linear               RMSE=1227.8162   MAE=823.8220


***
## Part 4 — Building an Exogenous-Aware Lag Matrix

### The key idea

On Day 2 you built a *multivariate* lag matrix using lagged values of a second series.
Today's exogenous variables are different in one important way: **we may know today's
weather today** — we don't need to lag it. A forecaster predicting tomorrow's rentals
could plausibly use *today's* temperature reading directly, since it's already observed
by the time the forecast is made.

This gives us a richer feature row:

$$
\mathbf{x}_t = \big[\, \underbrace{y_{t-1}, y_{t-2}, \ldots, y_{t-p}}_{\text{lags of the target}},\ \underbrace{e_t}_{\text{exogenous, same day}} \,\big]
$$

where $e_t$ is the exogenous variable's value on the *same day* as the target $y_t$
we're predicting — not lagged, since it's assumed known in advance (e.g., a weather
forecast).

### 4.1 Build the exogenous-aware lag matrix

**Your turn!** Complete the function below. It should produce the same lag columns as
`make_lag_matrix`, plus one additional column for the chosen exogenous variable —
**aligned to the same time index as the target**, not lagged.


In [ ]:
def make_lag_matrix_exog(series, exog, n_lags):
    """
    Build a lag-embedded feature matrix that includes lags of `series` PLUS
    the same-day value of an exogenous variable `exog`.

    Parameters
    ----------
    series : array-like, shape (T,)   the target series
    exog   : array-like, shape (T,)   the exogenous variable, aligned index-for-index
                                        with `series`
    n_lags : int, number of lag features for the target

    Returns
    -------
    X : np.ndarray, shape (T - n_lags, n_lags + 1)
    y : np.ndarray, shape (T - n_lags,)
    """
    series = np.array(series)
    exog   = np.array(exog)
    T = len(series)
    X, y = [], []
    for t in range(n_lags, T):
        lag_features = series[t - n_lags : t]
        # FILL IN: get the exogenous value at the SAME time index as the target (t)
        exog_today = exog[t]
        # FILL IN: combine lag_features and exog_today into one row
        # (hint: np.append(lag_features, exog_today))
        row = np.append(lag_features, exog_today)
        X.append(row)
        y.append(series[t])
    return np.array(X), np.array(y)


### 4.2 Apply it using `Temp` as the exogenous variable

**Your turn!** You'll need the exogenous series aligned to the same rows as
`Rentals_Diff` — remember that differencing dropped the first row.


In [15]:
# Align Temp to the same index range as Rentals_Diff (which lost its first row to .diff())
temp_aligned = df['Temp'].iloc[1:].values   # drop first row to match Rentals_Diff

# FILL IN: call make_lag_matrix_exog with target_series, temp_aligned, and N_LAGS
X_exog, y_exog = make_lag_matrix_exog(target_series, temp_aligned, N_LAGS)

print('Exogenous-aware X shape:', X_exog.shape)
print('(Should have one more column than the univariate X_uni.)')


Exogenous-aware X shape: (723, 8)
(Should have one more column than the univariate X_uni.)


### 4.3 Split and fit

**Your turn!** Apply the same chronological split logic as Part 3, then fit a second
`LinearRegression` on the exogenous-aware data.


In [ ]:
# FILL IN: chronological split, same TRAIN_FRAC as before
split_exog = int(len(X_exog) * TRAIN_FRAC)

X_exog_train, X_exog_test = X_exog[:split_exog], X_exog[split_exog:]
y_exog_train, y_exog_test = y_exog[:split_exog], y_exog[split_exog:]

lr_exog = LinearRegression()
#lr_exog.fit(???, ???)
#lr_exog_preds = lr_exog.predict(???)

#results['Exogenous (lags + Temp)'] = compute_metrics(y_exog_test, lr_exog_preds, 'Exogenous Linear')


### ✏️ Written Response 4

1. Did adding `Temp` improve RMSE and MAE relative to the univariate model? By how much?
2. Look at `lr_exog.coef_` — the last coefficient corresponds to `Temp`. What sign is
   it, and does that match the correlation you found in Part 1.3?
3. If the improvement was small, propose one reason why temperature alone might not
   add much beyond what the rental series' own recent history already captures.

> **YOUR ANSWER:**
